<a href="https://colab.research.google.com/github/shutashimizu/AI/blob/main/%E3%83%81%E3%83%A3%E3%83%BC%E3%83%88%E3%83%91%E3%82%BF%E3%83%BC%E3%83%B3%E4%BA%88%E6%83%B3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Flask Webアプリで画像アップロード


#Googleドライブのマウント
保存済みの重み付きモデルを取り出すため。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## PyTorch関連モジュールのインポート


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms, models
import torch.utils.data as data

In [ ]:
import  os
import glob
import re
from PIL import Image
import matplotlib.pyplot as plt

## 各種設定

In [ ]:
BASE_PATH = "/content/drive/MyDrive/MLData/"
DATASET_NAME = "images_fx"
LABEL_LIST = ["daburubotomu", "daburutop", "hedbotomu", "hedtop"]
#DATASET_NAME = " images_fx"
#LABEL_LIST = [ "daburubotomu", "daburutop","hedbotomu","hedtop","jousyousankaku","kakousankaku" ]
CHECKPOINT_PATH = BASE_PATH+DATASET_NAME+"_checkpoints"
#入力画像の大きさ
IMAGE_SIZE = 224

#判別結果の保存先
predicts = {}

In [ ]:
test_transforms = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
    ]
)

##重み付きモデルの読み込み
torch.load( ファイル名 ) を用いる

In [ ]:
# googleドライブのMLDataのimages_dogcatにmodel_???.pthがあることを確認
#MODEL_NAME = "cnn5fc2"
MODEL_NAME = "ResNet152"

In [ ]:
model = torch.load(BASE_PATH + DATASET_NAME + "/"+ "model_"+MODEL_NAME+".pth", map_location=torch.device('cpu'))

##予測関数の用意
予測部分を関数化する

In [ ]:
def predict_image(filename):
    image = Image.open(filename)
    image = image.convert("RGB")
    image = test_transforms(image)
    images = image.unsqueeze(0)

    model.eval()
    predict_results = torch.sigmoid(model(images))
    n = predict_results[0].argmax()

    return LABEL_LIST[n], predict_results[0][n]

# Webページテンプレート生成

## HTML template

Home

In [ ]:
# 追加するホーム画面用のHTMLファイル作成
dirname = 'templates'
filename = 'index.html'
contents = """
<!DOCTYPE html>
<html>
  <head>
    <meta charset="utf-8">
    <title>Image Collecter</title>
    <meta name="viewport" content="width=device-width,initial-scale=1.0,minimum-scale=1.0">
    <link href="/static/style.css" rel="stylesheet" type="text/css">
  </head>
  <body>
    <div class="container">
      <h1>チャートパターン予想</h1>
      <p>画像をアップロードして予測を始めよう！</p>
      <p><a href="/upload.html" class="btn">始める</a></p>
    </div>
  </body>
</html>
"""
if not os.path.isdir(dirname):
    os.makedirs(dirname)
fp = open(dirname + '/' + filename, mode='w')
fp.write(contents)
fp.close()


### list.html

In [ ]:
"""
アップロードした画像を一覧表示するための HTML
画像が無いときもこの HTML を表示
"""
dirname = 'templates'
filename = 'list.html'
contents = """
<!DOCTYPE html>
<html>
  <head>
    <meta charset="utf-8">
    <title>Image Collecter</title>
    <meta name="viewport" content="width=device-width,initial-scale=1.0,minimum-scale=1.0">
    <link href="/static/style.css" rel="stylesheet" type="text/css">
  </head>
  <body>
    <div class="container">
      <h1>分析結果</h1>
      <p><a href="/upload.html" class="btn">画像登録</a></p>
      <p>登録済み画像一覧</p>
      <div class="image-gallery">
        {% for img in imglst %}
          <div class="imageBox">
            <a href="image/{{ img.filename }}">
              <img src="/static/{{ img.filename }}" alt="{{ img.filename }}" class="imageThumbnail">
            </a>
            <div class="imageTitle">{{ img.classname }}</div>
            <!-- 帯グラフ -->
            <div class="progress-bar-container">
              <div class="progress-bar" style="width: {{ img.score }}%;"></div>
            </div>
            <!-- パーセンテージ表示 -->
            <div class="percentage-text">{{ img.score }}%</div>
            <div class="filename">{{ img.filename }}</div>
          </div>
        {% endfor %}
      </div>
    </div>
  </body>
</html>

"""

import  os
if not os.path.isdir(dirname):
    os.makedirs(dirname)
fp = open(dirname + '/' + filename, mode='w')
fp.write(contents)
fp.close()

### image.html

In [ ]:
"""
アップロードした画像を一覧表示するための HTML
画像が無いときもこの HTML を表示
"""
dirname = 'templates'
filename = 'image.html'
contents = """
<!DOCTYPE html>
<html>
  <head>
    <meta charset="utf-8">
    <title>Image Collecter</title>
    <meta name="viewport" content="width=device-width,initial-scale=1.0,minimum-scale=1.0">
    <link href="/static/style.css" rel="stylesheet" type="text/css">
  </head>
  <body>
    <div class="container">
      <h1>個別画面</h1>
      <!-- 戻るリンクを /list に変更 -->
      <p><a href="/list" class="btn">戻る</a></p>
      <img src="/static/{{ name }}" alt="{{ name }}" class="full-image">
      <!-- 再度戻るリンクを /list に変更 -->
      <p><a href="/list" class="btn">戻る</a></p>
    </div>
  </body>
</html>


"""

import  os
if not os.path.isdir(dirname):
    os.makedirs(dirname)
fp = open(dirname + '/' + filename, mode='w')
fp.write(contents)
fp.close()

### upload.html

In [ ]:
"""
アップロードした画像を一覧表示するための HTML
画像が無いときもこの HTML を表示
"""
dirname = 'templates'
filename = 'upload.html'
contents = """
<!DOCTYPE html>
<html>
  <head>
    <meta charset="utf-8">
    <title>Image Collecter</title>
    <meta name="viewport" content="width=device-width,initial-scale=1.0,minimum-scale=1.0">
    <link href="/static/style.css" rel="stylesheet" type="text/css">
    <script>
      // 画像ファイル選択時にアップロードボタンを表示
      function handleFileSelect(event) {
        var uploadButton = document.getElementById('uploadButton');
        var fileInput = event.target;
        if (fileInput.files.length > 0) {
          uploadButton.style.display = 'block';  // ファイルが選択されたらボタンを表示
        } else {
          uploadButton.style.display = 'none';  // ファイルが選択されていなければボタンを隠す
        }
      }
    </script>
  </head>
  <body>
    <div class="container">
      <h1>チャートをアップロードしてね！</h1>
      <p><a href="/list">画像一覧</a></p>
      <form action="/upload" method="post" enctype="multipart/form-data">
        <label for="file_content">画像ファイルを選択：</label><br>
        <input type="file" name="file_content" accept="image/*" onchange="handleFileSelect(event)"><br><br>

        <!-- アップロードボタンは初期状態では非表示 -->
        <input type="submit" value="アップロード" id="uploadButton" style="display:none;">
      </form>
    </div>
  </body>
</html>

"""

import  os
if not os.path.isdir(dirname):
    os.makedirs(dirname)
fp = open(dirname + '/' + filename, mode='w')
fp.write(contents)
fp.close()

## CSS

### style.css

In [ ]:
"""
HTML を表示するときのレイアウトやデザインを指定する
"""
dirname = 'static'
filename = 'style.css'
contents = """
/* 全体のレイアウト */
body {
  font-family: Arial, sans-serif;
  background-color: #f4f4f9;
  margin: 0;
  padding: 0;
  display: flex;
  justify-content: center;
  align-items: center;
  min-height: 100vh;
}

/* ホーム画面用スタイル */
.container {
  text-align: center;
  margin-top: 0;
  display: flex;
  flex-direction: column;
  justify-content: center;
  align-items: center;
  width: 100%;
  min-height: 100vh; /* ビューポートの高さを使って中央配置 */
}

h1 {
  font-size: 36px;
  font-weight: bold;
  color: #007BFF;
  margin-bottom: 20px;
}

p {
  font-size: 18px;
  color: #333;
}

.btn {
  background-color: #007BFF;
  padding: 15px 30px;
  color: white;
  font-size: 18px;
  text-decoration: none;
  border-radius: 5px;
  display: inline-block;
}

.btn:hover {
  background-color: #0056b3;
}

/* 個別画像ページの画像スタイル */
.full-image {
  max-width: 90%; /* 画面幅の90%以内に収める */
  max-height: 80vh; /* ビューポートの高さの80%以内に収める */
  margin: auto;
  display: block; /* 中央揃え */
  border: 2px solid #ddd; /* 軽い枠線を追加 */
  border-radius: 5px; /* 角を丸める */
}

/* メインコンテナ */
.container {
  width: 90%;
  max-width: 1200px;
  margin: 0 auto;
  padding: 20px;
  box-sizing: border-box; /* パディングを含む幅を計算 */
}

/* タイトル */
h1 {
  padding: 0.5em 1em;
  background: linear-gradient(to right, #74b6ff, #4a8ed9); /* さらに少し濃い青グラデーション */
  color: white;
  text-align: center;
  border-radius: 5px;
  box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1);
}

/* 登録ボタン */
.btn {
  background-color: #28a745;
  padding: 10px 20px;
  color: white;
  border-radius: 5px;
  text-decoration: none;
}

.btn:hover {
  background-color: #218838;
}

/* 画像ギャラリー */
.image-gallery {
  display: flex;
  flex-wrap: wrap;
  gap: 20px;
  justify-content: space-around;
  max-height: 60vh; /* ギャラリーの最大高さを指定 */
  overflow-y: auto; /* 高さがオーバーフローした場合、スクロールできるように */
  width: 100%; /* ギャラリーの幅を100%に */
  box-sizing: border-box; /* ギャラリー内の余白を計算 */
  padding: 10px;
}

/* 画像ボックス */
.imageBox {
  background-color: #fff;
  box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
  border-radius: 8px;
  overflow: hidden;
  width: 240px;
  text-align: center;
  transition: transform 0.3s, box-shadow 0.3s;
}

.imageBox:hover {
  transform: scale(1.05);
  box-shadow: 0 6px 12px rgba(0, 0, 0, 0.2);
}

/* 画像のサムネイル */
.imageThumbnail {
  width: 100%;
  height: auto;
  border-bottom: 2px solid #eee;
  object-fit: cover;
}

/* 画像のタイトル */
.imageTitle {
  padding: 10px;
  font-weight: bold;
  background-color: #f8f9fa;
  border-top: 1px solid #ddd;
  color: #333;
}

/* ファイル名のスタイル */
.filename {
  font-size: 12px;
  color: #888;
  padding: 5px 0;
  background-color: #f8f9fa;
  border-top: 1px solid #ddd;
}

/* 帯グラフの外枠 */
.progress-bar-container {
  width: 100%; /* 横幅を100%にする */
  height: 20px; /* 高さ */
  background-color: #f0f0f0; /* 背景色（グレー） */
  border-radius: 10px; /* 丸みを指定 */
  margin-top: 5px; /* 少し余白を追加 */
  overflow: hidden; /* 内部の角を丸める */
  border: 1px solid #ddd; /* 枠線 */
}

/* 帯グラフの色付き部分 */
.progress-bar {
  height: 100%; /* 外枠の高さに合わせる */
  background-color: #007BFF; /* プログレスバーの色（青） */
  border-radius: 10px; /* 丸みを指定 */
  transition: width 0.3s ease-in-out; /* 幅が変化する際にアニメーションを追加 */
}

/* パーセンテージのテキスト */
.percentage-text {
  text-align: center; /* 中央揃え */
  font-size: 14px;
  color: #333; /* 濃いグレー */
  margin-top: 5px;
}

/* アップロードボタンを中央に配置 */
input[type="submit"] {
  margin-top: 20px; /* ボタンと入力欄の間に余白を追加 */
  padding: 15px 30px; /* ボタンの大きさを調整 */
  background-color: #28a745;
  color: white;
  font-size: 18px;
  border-radius: 5px;
  border: none;
  display: block; /* ボタンを中央に配置 */
  margin-left: auto;
  margin-right: auto;
}

input[type="submit"]:hover {
  background-color: #218838;
}

/* 画像一覧ページ用のボタン */
.image-gallery .btn {
  margin: 3px; /* 画像一覧ページのボタン間のマージンを更に狭くする */
}

/* レスポンシブデザイン */
@media (max-width: 768px) {
  .imageBox {
    width: 45%;
  }
}

@media (max-width: 480px) {
  .imageBox {
    width: 100%;
  }
}


"""

import  os
if not os.path.isdir(dirname):
    os.makedirs(dirname)
fp = open(dirname + '/' + filename, mode='w')
fp.write(contents)
fp.close()

# アプリケーションサーバ
Python と Flask で作成したプログラムコード

## proxy の設定
localhost のポート 5000 を外部に公開する proxy を求める。実行結果は localhost:5000 と表示されるが、実際の URL は異なっている点に注意。Flask の Web サーバを起動してから、ここで得られた URL の末尾にディレクトリ名やファイル名を追加してアクセスする。

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(5000, path='')

## プログラムメイン部分


In [ ]:
import os
import glob
import re
import base64
import datetime

from flask import Flask, render_template, request, redirect, url_for, jsonify

app = Flask(__name__)

# 画像保存フォルダ（static）から画像の一覧を作る関数
# 対象となる画像ファイル（拡張子：jpg,jpeg,png）のリストを作成
def get_list():
    print('>>> get_list() called:')
    imglst = [os.path.basename(fn) for fn in glob.glob('static/*')
              if re.search(r'\.(jpg|jpeg|png|JPG|JPEG|PNG)$', fn)]

    print('imglst: ', imglst)

    imgdata = []
    for imgfile in imglst:
        try:
            # スコアをテンソルからパーセントに変換 (四捨五入して小数点1桁にする)
            score = predicts[imgfile][1].item() * 100
            rounded_score = round(score, 1)  # 例: 96.6003 → 96.6
            imgdata.append({
                'filename': imgfile,
                'classname': predicts[imgfile][0],
                'score': rounded_score
            })
        except KeyError:
            imgdata.append({'filename': imgfile, 'classname': 'unknown', 'score': 0.0})

    return imgdata


# サーバのトップURLに対して返すホームページの生成
@app.route('/')
def home():
    print('>>> home() called:')
    return render_template('index.html')

# 画像一覧ページのURLを'/list'に変更
@app.route('/list')
def list():
    print('>>> list() called:')
    return render_template('list.html', imglst=get_list())


# 指定した画像をひとつだけ表示する
@app.route('/image/<filename>')
def image(filename):
    # 選択した画像だけ表示する
    # 単独の画像表示用テンプレートHTMLファイルと、表示したい画像ファイル名を
    # 引数として、render_template() を呼び出す
    # render_template(＜テンプレートファイル＞, ＜テンプレートに渡す画像ファイル名＞)
    print('>>> image('+filename+') called:')
    return render_template('image.html', name=filename)

# アップロードページを表示
@app.route('/upload.html')
def uploadPage():
    print('>>> uploadPage() called:')
    return render_template('upload.html')

@app.route('/upload', methods=['POST'])
def regist():
    # 画像を登録する
    # 登録画像は jpeg、png 形式に限定
    # ただし、ここでファイル種別のチェック処理はやっていない
    print('>>> regist() called:')

    # 画像データを取り出す
    file = request.files['file_content']

    # アップロードされた画像データを受け取って ./static フォルダへ保存
    file.save(os.path.join('./static', file.filename))

    #ファイルから予測
    predicts[file.filename] = predict_image(os.path.join('./static', file.filename))

    # 登録済み画像一覧ページを表示
    return render_template('list.html', imglst=get_list())

import os

# static フォルダ内の画像ファイル（.jpg, .jpeg, .png）だけ削除する
def clear_image_files_in_static():
    folder_path = './static'
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if os.path.isfile(file_path) and re.search(r'\.(jpg|jpeg|png|JPG|JPEG|PNG)$', filename):
            os.remove(file_path)

clear_image_files_in_static()  # 画像ファイルのみ削除


if __name__ == '__main__':
    app.run()